# 03 Agents & Tools

By the end you can define `@tool` schemas, run `create_agent()` (a CompiledStateGraph), and read the full ReAct loop in DATAFLOW.


Set up the project path and reload shared helpers from this repo.


In [ ]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Pick up edits to shared/ without a full kernel restart (re-run this cell)
for _name in (
    "shared.dataflow",
    "shared.notebook_display",
    "shared.llm",
    "shared.bootcamp_fixtures",
):
    importlib.reload(importlib.import_module(_name))

print(f"Project root: {ROOT}")


Check which API keys and provider settings are available in `.env`.


In [ ]:
import os

def env_status():
    keys = {
        "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
        "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "deepseek"),
        "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
        "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    }
    for k, v in keys.items():
        print(f"{k}: {v}")
    return keys

ENV = env_status()


Load the chat model helper used by live cells below.


In [ ]:
import os
from shared.llm import get_model

def require_llm():
    return get_model()


Import DATAFLOW printers once, they label each agent and RAG step so you read a run like a wizard tracing a spell. Later cells call print_agent_dataflow or print_rag_dataflow; watch those blocks closely.


In [ ]:
from shared.dataflow import preview, print_agent_dataflow, print_rag_dataflow, print_final_state


Print the JSON schema the model sees for search_destinations. Field descriptions become tool instructions, vague text means vague tool calls. Notice query and budget properties with descriptions in the output.


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool
import json

class SearchArgs(BaseModel):
    query: str = Field(description="Trip type: beach, city, nature")
    budget: int = Field(description="Max USD", ge=0)

@tool(args_schema=SearchArgs)
def search_destinations(query: str, budget: int) ->  str:
    """Search destinations matching query and budget."""
    return "Phuket, $450"

print(json.dumps(search_destinations.args_schema.model_json_schema(), indent=2))


Load shared travel tools and invoke them directly, no LLM yet. This shows the ToolMessage shape before the agent wraps them. Search and weather lines should print concrete strings.


In [ ]:
from shared.bootcamp_fixtures import (
    TRAVEL_TOOLS,
    check_weather,
    print_messages,
    search_destinations,
)
print("Tools:", [t.name for t in TRAVEL_TOOLS])

print("Search:", search_destinations.invoke({"query": "beach", "budget": 600}))
print("Weather:", check_weather.invoke({"city": "Phuket", "date": "2026-07-01"}))


Requires `DEEPSEEK_API_KEY` in `.env` (default provider: DeepSeek).


In [ ]:
from langchain.agents import create_agent

model = require_llm()
agent = create_agent(
    model=model,
    tools=[search_destinations, check_weather],
    system_prompt="Travel assistant. Check weather before recommending.",
)
print("Agent graph type:", type(agent).__name__)
question = "Beach under $600, weather in Phuket?"
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print_agent_dataflow(result["messages"])
